# XGBoost Model

In [ ]:
!pip install xgboost pandas numpy scikit-learn joblib

**Importing Libraries**
                                                  
This code is used to load all the necessary tools needed for the project.Using this code, we can build a prediction model and check how accurate its results are. It helps us save the trained model so it can be reused in the future without training again.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
import joblib

**Loading Dataset**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Cleaned_agriculture_price_dataset.csv')
df.head()

,price_date,state,district_name,market_name,commodity,variety,min_price,max_price,modal_price
0,2023-06-06,Maharashtra,Nashik,Lasalgaon(Niphad),Wheat,Maharashtra 2189,2172.0,2399.0,2300.0
1,2023-06-06,Uttar Pradesh,Bijnor,Chaandpur,Tomato,Hybrid,600.0,700.0,650.0
2,2023-06-06,Jammu & Kashmir,Jammu,Batote,Tomato,Other,1800.0,2200.0,2000.0
3,2023-06-06,Gujarat,Dahod,Dahod,Wheat,147 Average,2500.0,2700.0,2600.0
4,2023-06-06,Madhya Pradesh,Guna,Guna(F&V),Tomato,Other,350.0,530.0,410.0


**Inference:**
This step is used to load the dataset into the notebook so that it can be analyzed and processed. After loading, the data becomes available in a table format, which makes it easy to view, clean, and use for further operations like analysis and model building.

**Date Formatting, Data Sorting and Final Data Validation**


In [ ]:
df['price_date'] = pd.to_datetime(df['price_date'])

df = df.sort_values(by=['market_name', 'commodity', 'price_date'])

df = df.dropna(subset=['modal_price'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 695213 entries, 779 to 184101
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   price_date     695213 non-null  datetime64[ns]
 1   state          695213 non-null  object        
 2   district_name  695213 non-null  object        
 3   market_name    695213 non-null  object        
 4   commodity      695213 non-null  object        
 5   variety        695213 non-null  object        
 6   min_price      695213 non-null  float64       
 7   max_price      695213 non-null  float64       
 8   modal_price    695213 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(5)
memory usage: 53.0+ MB


In [ ]:
df['day'] = df['price_date'].dt.day
df['month'] = df['price_date'].dt.month
df['year'] = df['price_date'].dt.year
df['day_of_week'] = df['price_date'].dt.dayofweek

**Inference:**
The dataset is prepared for reliable analysis by converting the price date column into a proper datetime format.The data is then systematically sorted based on market name, commodity, and date to maintain a clear chronological order.The dataset structure is verified to confirm correct data types, non-null values, and overall data integrity.


**Creating Lagged Price Features**


Lagged features help the model understand how past prices influence current prices. The 1-day lag captures immediate price changes, while the 7-day and 14-day lags reflect weekly and bi-weekly trends. The 30-day lag provides insight into monthly price behavior.

In [ ]:
df['lag_1']  = df.groupby(['market_name','commodity'])['modal_price'].shift(1)
df['lag_7']  = df.groupby(['market_name','commodity'])['modal_price'].shift(7)
df['lag_14'] = df.groupby(['market_name','commodity'])['modal_price'].shift(14)
df['lag_30'] = df.groupby(['market_name','commodity'])['modal_price'].shift(30)

In [ ]:
df['rolling_mean_7']  = df.groupby(['market_name','commodity'])['modal_price'].shift(1).rolling(7).mean()
df['rolling_mean_14'] = df.groupby(['market_name','commodity'])['modal_price'].shift(1).rolling(14).mean()

**Removing Missing Values and Checking Dataset Shape**

After dropping rows with missing values, the dataset becomes cleaner and more reliable. The updated shape confirms the final size of the dataset, which is now suitable for further analysis and visualization.

In [ ]:

df = df.dropna()
df.shape

(605890, 19)

**Label Encoding of Categorical Columns**

In this step, categorical columns such as state, district_name, market_name, and commodity are converted into numerical values using Label Encoding. This transformation is necessary because machine learning models cannot work directly with text data.



In [ ]:
label_encoders = {}
cat_cols = ['state', 'district_name', 'market_name', 'commodity']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

**Feature Selection for Price Prediction**

By combining geographical, temporal, and historical price-based features, the dataset becomes well-structured fortime-series-based price prediction. Lag features and rolling averages play a crucial role in capturing past price behavior, while date-related features help identify seasonal trends.


In [ ]:
features = [
    'state','district_name','market_name','commodity',
    'day','month','year','day_of_week',
    'lag_1','lag_7','lag_14','lag_30',
    'rolling_mean_7','rolling_mean_14'
]

X = df[features]
y = df['modal_price']

**Train–Test Split Based on Date**

In this step,the dataset is split into training and testing sets based on the price_date  column.

In [ ]:
split_date = df['price_date'].quantile(0.8)

X_train = X[df['price_date'] <= split_date]
X_test  = X[df['price_date'] > split_date]

y_train = y[df['price_date'] <= split_date]
y_test  = y[df['price_date'] > split_date]

**XGBoost Regression Model Training**

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

**Inference:**
XGBoost efficiently handles regression tasks with high accuracy.Using these hyperparameters, the model balances bias and variance.
It Suitable for predicting continuous values like prices, sales, or measurements.

**Model Prediction and Evaluation**

This step ensures that the trained model and encoders can be reused later without retraining. It is especially useful for deploying the model or making predictions on new data.

In [ ]:
preds = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))

MAE : 159.28240735597586
RMSE: 268.5107723488325


In [ ]:
joblib.dump(model, '/content/drive/MyDrive/xgboost_model.pkl') #do not run
joblib.dump(label_encoders, '/content/drive/MyDrive/label_encoders.pkl')

['/content/drive/MyDrive/label_encoders.pkl']

**Encoding Input Features**

This function encode_inputs takes categorical input values like state district market and commodity and converts them into numerical representations using pre-fitted label encoders. The function returns a dictionary containing the encoded values ready for model prediction.

In [ ]:
def encode_inputs(state, district, market, commodity):
    return {
        'state': label_encoders['state'].transform([state])[0],
        'district_name': label_encoders['district_name'].transform([district])[0],
        'market_name': label_encoders['market_name'].transform([market])[0],
        'commodity': label_encoders['commodity'].transform([commodity])[0]
    }

**Predict Future Commodity Prices**

This function predicts future commodity prices for a specified state, district, market, and commodity using a trained machine learning model. The function returns a DataFrame of predicted prices, updating features after each prediction, or a message if no historical data is available.

In [ ]:
def predict_future_prices(df, model, state, district, market, commodity, days):

    import pandas as pd

    enc = encode_inputs(state, district, market, commodity)

    hist = df[
        (df['market_name'] == enc['market_name']) &
        (df['commodity'] == enc['commodity'])
    ].sort_values('price_date')

    if hist.empty:
        return "No historical data available for given inputs."

    # 🔥 Use last known row only for lag features
    last = hist.iloc[-1:].copy()

    # 🔥 START FROM TODAY (not dataset date)
    last_date = pd.Timestamp.today().normalize()

    results = []

    for _ in range(days):
        next_date = last_date + pd.Timedelta(days=1)

        # Date features
        last['day'] = next_date.day
        last['month'] = next_date.month
        last['year'] = next_date.year
        last['day_of_week'] = next_date.dayofweek

        X_next = last[features]
        price = model.predict(X_next)[0]

        results.append({
            'price_date': next_date,
            'predicted_price': round(price, 2)
        })

        # Update lag features (autoregressive)
        last['lag_30'] = last['lag_14']
        last['lag_14'] = last['lag_7']
        last['lag_7'] = last['lag_1']
        last['lag_1'] = price

        # Update rolling features
        last['rolling_mean_7']  = (last['rolling_mean_7'] * 6 + price) / 7
        last['rolling_mean_14'] = (last['rolling_mean_14'] * 13 + price) / 14

        last_date = next_date

    return pd.DataFrame(results)


**Predicting Future Commodity Prices**

In [ ]:
market = input("Enter market name: ")
district = input("Enter district name: ")
state = input("Enter state: ")
commodity = input("Enter commodity: ")
days = int(input("Enter number of days to predict: "))
future_prices = predict_future_prices(
    df=df,
    model=model,
    state=state,
    district=district,
    market=market,
    commodity=commodity,
    days=days
)

print(future_prices)

Enter market name: Batote
Enter district name: Jammu
Enter state: Jammu & Kashmir
Enter commodity: Tomato
Enter number of days to predict: 7
  price_date  predicted_price
0 2025-12-23      1994.130005
1 2025-12-24      1922.869995
2 2025-12-25      1878.430054
3 2025-12-26      1867.369995
4 2025-12-27      1856.140015
5 2025-12-28      1853.489990
6 2025-12-29      1856.339966


**Inference:**
The predicted prices of Tomato in Batote market, Jammu district, Jammu & Kashmir for the next 7 days show a gradual decrease from 1994.13 to 1856.34, indicating a potential downward trend in the market price over the week.Users can use this forecast to make informed decisions regarding buying, selling, or storing commodities.

##CONCLUSION

**Data Cleaning & Preprocessing**

Handled missing values, formatted columns, and encoded categorical features.

**Exploratory Data Analysis**

Identified key patterns and relationships among features and the target variable.

**Model Building**

Implemented XGBoost, a robust machine learning algorithm for prediction.

**Model Evaluation**

Assessed performance using MAE and MSE, showing reliable and accurate results.

**Key Takeaway**

Demonstrated the end-to-end workflow: data cleaning → analysis → modeling → evaluation → model saving for future use.

**Overall**

The entire workflow can be executed sequentially to replicate the results:  
1. Data cleaning & preprocessing  
2. Exploratory analysis  
3. Model building with XGBoost  
4. Model evaluation and saving
